# Jane Street Conservative RLS Late Submission

Attach the official competition data and the private Dataset containing `jane-street-conservative-rls-late-submission`. Keep internet disabled and use GPU if available. Submit the saved Kaggle Notebook version, not a local parquet or zip file.

In [ ]:
from pathlib import Path
import os
import runpy
import shutil
import zipfile


def is_package_dir(path: Path) -> bool:
    return (
        (path / 'submission/submission.py').exists()
        and (path / 'artifacts/jane_street_submission/base_models/manifest.json').exists()
        and (path / 'artifacts/jane_street_submission/meta_rls_experts_alpha10000_f0p995/rls_state.npz').exists()
    )


def find_package_dir() -> Path:
    input_root = Path('/kaggle/input')
    direct_matches = []
    for submission_file in sorted(input_root.rglob('submission/submission.py')):
        candidate = submission_file.parents[1]
        if is_package_dir(candidate):
            direct_matches.append(candidate)
    if direct_matches:
        print('Using extracted package:', direct_matches[0])
        return direct_matches[0]

    for zip_path in sorted(input_root.rglob('*.zip')):
        with zipfile.ZipFile(zip_path) as archive:
            names = archive.namelist()
            if not any(name.endswith('submission/submission.py') for name in names):
                continue
            extract_root = Path('/kaggle/working/jane_street_conservative_package')
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True, exist_ok=True)
            archive.extractall(extract_root)
        for submission_file in sorted(extract_root.rglob('submission/submission.py')):
            candidate = submission_file.parents[1]
            if is_package_dir(candidate):
                print('Extracted package from:', zip_path)
                return candidate

    print('Available /kaggle/input entries:', sorted(str(path) for path in input_root.glob('*')))
    raise FileNotFoundError('Could not find the conservative RLS package as extracted files or as a zip archive.')


package_dir = find_package_dir()
for competition_dir in (
    Path('/kaggle/input/jane-street-real-time-market-data-forecasting'),
    Path('/kaggle/input/competitions/jane-street-real-time-market-data-forecasting'),
):
    if (competition_dir / 'test.parquet').exists() or (competition_dir / 'test.parquet').is_dir():
        os.environ.setdefault('JANE_STREET_COMPETITION_DIR', str(competition_dir))
        print('Using competition data:', competition_dir)
        break
os.environ.setdefault('JANE_STREET_BASE_ARTIFACT_DIR', str(package_dir / 'artifacts/jane_street_submission/base_models'))
os.environ.setdefault(
    'JANE_STREET_META_ARTIFACT_DIR',
    str(package_dir / 'artifacts/jane_street_submission/meta_rls_experts_alpha10000_f0p995'),
)
os.environ.setdefault('JANE_STREET_DEVICE', 'cuda')
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    os.environ.setdefault('JANE_STREET_RUN_LOCAL_GATEWAY', '1')
print('Running submission from package:', package_dir)
runpy.run_path(str(package_dir / 'submission/submission.py'), run_name='__main__')
